# Non-linear C2ST: RBF SVM

In [2]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

In [3]:
import importlib
import src.metrics as metrics

importlib.reload(metrics)

from src.metrics import c2st_rbf_paired

In [4]:
# monte carlo helpers
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

N=100
REPS=200
ALPHA=0.05
BASE_SEED=42

def run_rbf_power(clean,conditions,scenario,n=N,reps=REPS,base_seed=BASE_SEED):
    rows=[]
    n_obs=clean.shape[0]

    if n>n_obs:
        raise ValueError(f"Requested n={n}, but only {n_obs} observations are available.")

    for rep in range(reps):
        seed=base_seed+rep
        idx=np.random.default_rng(seed).choice(n_obs,size=n,replace=False)

        for severity,Qall in conditions.items():
            P=clean[idx]
            Q=Qall[idx]

            r=c2st_rbf_paired(
                P,
                Q,
                seed=seed,
                alpha=ALPHA
            )

            rows.append({
                "scenario":scenario,
                "severity":float(severity),
                "repetition":rep,
                **r
            })

    return pd.DataFrame(rows)


def run_rbf_null(clean,null_name,n=N,reps=REPS,base_seed=BASE_SEED):
    rows=[]
    n_obs=clean.shape[0]

    if 2*n>n_obs:
        raise ValueError(f"Need at least {2*n} observations, but only {n_obs} are available.")

    for rep in range(reps):
        seed=base_seed+rep
        idx=np.random.default_rng(seed).choice(n_obs,size=2*n,replace=False)

        P=clean[idx[:n]]
        Q=clean[idx[n:]]

        r=c2st_rbf_paired(
            P,
            Q,
            seed=seed,
            alpha=ALPHA
        )

        rows.append({
            "null":null_name,
            "repetition":rep,
            **r
        })

    return pd.DataFrame(rows)


def summarize_rejections(df,group_cols):
    out=df.groupby(group_cols)["reject_H0"].agg(["sum","count","mean"]).reset_index()

    ci=[
        proportion_confint(k,n,alpha=.05,method="wilson")
        for k,n in zip(out["sum"],out["count"])
    ]

    out["ci_low"]=[x[0] for x in ci]
    out["ci_high"]=[x[1] for x in ci]

    return out.rename(columns={
        "sum":"rejections",
        "count":"repetitions",
        "mean":"power"
    })

In [5]:
# project paths
from src.paths import get_project_paths

P=get_project_paths()
DATA_DIR=P["processed"]
RESULTS_DIR=P["results"]
W7=RESULTS_DIR/"week7"

print("DATA_DIR:",DATA_DIR)
print("W7:",W7)

DATA_DIR: /Users/jessica/fidelity-metric-lab/data/processed
W7: /Users/jessica/fidelity-metric-lab/results/week7


In [6]:
# inspect frozen artifacts
print("\nProcessed numpy files:")
for p in sorted(DATA_DIR.rglob("*.npy")): print(p.relative_to(DATA_DIR))

print("\nProcessed npz files:")
for p in sorted(DATA_DIR.rglob("*.npz")): print(p.relative_to(DATA_DIR))

print("\nWeek 7 numpy files:")
for p in sorted(W7.rglob("*.npy")): print(p.relative_to(W7))


Processed numpy files:
archive/entity_embeddings_stale_pre_week7/entity_embeddings_10pct.npy
archive/entity_embeddings_stale_pre_week7/entity_embeddings_30pct.npy
archive/entity_embeddings_stale_pre_week7/entity_embeddings_60pct.npy
archive/entity_embeddings_stale_pre_week7/entity_embeddings_clean.npy
entity_embeddings_10pct.npy
entity_embeddings_30pct.npy
entity_embeddings_60pct.npy
entity_embeddings_clean.npy
missing_reference_embeddings_10pct.npy
missing_reference_embeddings_30pct.npy
missing_reference_embeddings_60pct.npy
missing_reference_embeddings_clean.npy
over_polished_embeddings_10pct.npy
over_polished_embeddings_30pct.npy
over_polished_embeddings_60pct.npy
over_polished_embeddings_clean.npy
over_polished_reference_embeddings.npy

Processed npz files:
template_tfidf_10pct.npz
template_tfidf_30pct.npz
template_tfidf_60pct.npz
template_tfidf_clean.npz
template_tfidf_reference.npz

Week 7 numpy files:


In [7]:
print("Processed:")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file() and any(x in p.name.lower() for x in ["broken","timing","structural"]):
        print(p.relative_to(DATA_DIR))

print("\nWeek 7:")
for p in sorted(W7.rglob("*")):
    if p.is_file() and any(x in p.name.lower() for x in ["broken","timing","structural"]):
        print(p.relative_to(W7))

Processed:
broken_timing_formal_plan.csv
broken_timing_structural_10pct.csv
broken_timing_structural_30pct.csv
broken_timing_structural_60pct.csv
broken_timing_structural_clean.csv
clean_structural_eval_scaled.csv
clean_structural_summary.csv
clean_structural_train_scaled.csv
structural_campaign_split.csv

Week 7:
broken_timing_formal_configuration.json
broken_timing_formal_null_runs.csv
broken_timing_formal_null_summary.csv
broken_timing_formal_power_runs.csv
broken_timing_formal_power_summary.csv
broken_timing_formal_power_table.csv
broken_timing_formal_representation_response.csv
broken_timing_formal_sample_prevalence.csv
broken_timing_formal_sinkhorn_null_gap.csv
broken_timing_formal_sinkhorn_power_runs_enriched.csv


In [8]:
import pandas as pd

bt_check=pd.read_csv(DATA_DIR/"broken_timing_structural_clean.csv")
print(bt_check.shape)
print(bt_check.columns.tolist())
display(bt_check.head())

(300, 14)
['campaign_id', 'n_messages', 'campaign_duration', 'mean_time_gap', 'median_time_gap', 'std_time_gap', 'n_channels', 'channel_switch_rate', 'channel_entropy', 'dominant_channel_share', 'mean_message_length', 'std_message_length', 'mean_token_count', 'std_token_count']


,campaign_id,n_messages,campaign_duration,mean_time_gap,median_time_gap,std_time_gap,n_channels,channel_switch_rate,channel_entropy,dominant_channel_share,mean_message_length,std_message_length,mean_token_count,std_token_count
0,C0003,7.0,372.0,62.000000,57.0,16.552945,2.0,0.166667,0.863121,0.714286,60.428571,20.742346,11.142857,3.907084
1,C0004,7.0,349.0,58.166667,60.0,32.482901,3.0,0.666667,0.914101,0.428571,62.000000,19.566736,11.000000,3.854496
2,C0006,9.0,509.0,63.625000,71.0,30.646931,3.0,0.500000,0.905713,0.555556,56.000000,20.237479,9.111111,4.094561
3,C0008,9.0,479.0,59.875000,70.0,33.141505,3.0,0.875000,0.905713,0.555556,59.666667,19.037974,10.555556,3.715167
4,C0010,5.0,276.0,69.000000,67.0,15.049917,2.0,0.500000,0.970951,0.600000,64.600000,20.075856,12.000000,3.577709


In [9]:
import numpy as np
import pandas as pd

def load_structural_csv(path):
    df=pd.read_csv(path)
    df=df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")],errors="ignore")
    id_cols=[c for c in ["campaign_id","id"] if c in df.columns]
    ids=df[id_cols[0]].astype(str).to_numpy() if id_cols else None
    X=df.drop(columns=id_cols).to_numpy(dtype=float)
    return ids,X

bt_ids,bt_clean=load_structural_csv(DATA_DIR/"broken_timing_structural_clean.csv")
bt_ids10,bt_10=load_structural_csv(DATA_DIR/"broken_timing_structural_10pct.csv")
bt_ids30,bt_30=load_structural_csv(DATA_DIR/"broken_timing_structural_30pct.csv")
bt_ids60,bt_60=load_structural_csv(DATA_DIR/"broken_timing_structural_60pct.csv")

print(bt_clean.shape,bt_10.shape,bt_30.shape,bt_60.shape)

(300, 13) (300, 13) (300, 13) (300, 13)


In [10]:
assert bt_clean.shape==bt_10.shape==bt_30.shape==bt_60.shape
if bt_ids is not None:
    assert np.array_equal(bt_ids,bt_ids10)
    assert np.array_equal(bt_ids,bt_ids30)
    assert np.array_equal(bt_ids,bt_ids60)

print("Broken Timing alignment OK.")

Broken Timing alignment OK.


In [11]:
_,bt_eval=load_structural_csv(DATA_DIR/"clean_structural_eval_scaled.csv")

print("Same shape:",bt_clean.shape==bt_eval.shape)
print("Max abs difference:",np.max(np.abs(bt_clean-bt_eval)))

Same shape: True
Max abs difference: 705.2277783477878


In [12]:
# Validate the frozen Broken Timing formal representation
bt_files={
    "clean":DATA_DIR/"broken_timing_structural_clean.csv",
    .1:DATA_DIR/"broken_timing_structural_10pct.csv",
    .3:DATA_DIR/"broken_timing_structural_30pct.csv",
    .6:DATA_DIR/"broken_timing_structural_60pct.csv"
}

bt_dfs={k:pd.read_csv(p).drop(columns=[c for c in pd.read_csv(p,nrows=0).columns if c.startswith("Unnamed:")],errors="ignore") for k,p in bt_files.items()}

print("Columns identical:",all(list(df.columns)==list(bt_dfs["clean"].columns) for df in bt_dfs.values()))
print("Shapes:",{k:df.shape for k,df in bt_dfs.items()})

Columns identical: True
Shapes: {'clean': (300, 14), 0.1: (300, 14), 0.3: (300, 14), 0.6: (300, 14)}


In [13]:
# Keep campaign IDs separate from the 13 structural features
id_col="campaign_id" if "campaign_id" in bt_dfs["clean"].columns else None
feature_cols=[c for c in bt_dfs["clean"].columns if c!=id_col]

bt_ids=bt_dfs["clean"][id_col].astype(str).to_numpy() if id_col else None

timing={
    "clean":bt_dfs["clean"][feature_cols].to_numpy(dtype=float),
    .1:bt_dfs[.1][feature_cols].to_numpy(dtype=float),
    .3:bt_dfs[.3][feature_cols].to_numpy(dtype=float),
    .6:bt_dfs[.6][feature_cols].to_numpy(dtype=float)
}

print({k:X.shape for k,X in timing.items()})

{'clean': (300, 13), 0.1: (300, 13), 0.3: (300, 13), 0.6: (300, 13)}


In [14]:
# load semantic embeddings
from src.io_utils import load_embedding_conditions

entity=load_embedding_conditions(DATA_DIR,"entity",severities=(10,30,60))
missing=load_embedding_conditions(DATA_DIR,"missing_reference",severities=(10,30,60))
op=load_embedding_conditions(DATA_DIR,"over_polished",severities=(10,30,60))

print("Entity:",entity["clean"].shape)
print("Missing:",missing["clean"].shape)
print("Over-Polished:",op["clean"].shape)

Entity: (300, 384)
Missing: (300, 384)
Over-Polished: (300, 384)


In [15]:
# load frozen emplate tf idf matrices
from scipy.sparse import load_npz

template={
    "clean":load_npz(DATA_DIR/"template_tfidf_clean.npz"),
    .1:load_npz(DATA_DIR/"template_tfidf_10pct.npz"),
    .3:load_npz(DATA_DIR/"template_tfidf_30pct.npz"),
    .6:load_npz(DATA_DIR/"template_tfidf_60pct.npz")
}

print("Template clean:",template["clean"].shape)
print("Template conditions:",{s:X.shape for s,X in template.items() if s!="clean"})

Template clean: (300, 5000)
Template conditions: {0.1: (300, 5000), 0.3: (300, 5000), 0.6: (300, 5000)}


In [16]:
# load frozen Broken Timing structural representations


def load_bt_csv(path):
    df=pd.read_csv(path)
    df=df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")],errors="ignore")
    id_col="campaign_id" if "campaign_id" in df.columns else None
    ids=df[id_col].astype(str).to_numpy() if id_col else None
    X=df.drop(columns=[id_col] if id_col else []).to_numpy(dtype=float)
    return ids,X

bt_ids,bt_clean=load_bt_csv(DATA_DIR/"broken_timing_structural_clean.csv")
bt_ids10,bt_10=load_bt_csv(DATA_DIR/"broken_timing_structural_10pct.csv")
bt_ids30,bt_30=load_bt_csv(DATA_DIR/"broken_timing_structural_30pct.csv")
bt_ids60,bt_60=load_bt_csv(DATA_DIR/"broken_timing_structural_60pct.csv")

assert bt_clean.shape==bt_10.shape==bt_30.shape==bt_60.shape
if bt_ids is not None:
    assert np.array_equal(bt_ids,bt_ids10)
    assert np.array_equal(bt_ids,bt_ids30)
    assert np.array_equal(bt_ids,bt_ids60)

timing={"clean":bt_clean,.1:bt_10,.3:bt_30,.6:bt_60}

print("Broken Timing:",{k:X.shape for k,X in timing.items()})

Broken Timing: {'clean': (300, 13), 0.1: (300, 13), 0.3: (300, 13), 0.6: (300, 13)}


In [17]:
# build all five frozen scenarios
SCENARIOS={
    "Template Repetition":{
        "clean":template["clean"],
        "conditions":{.1:template[.1],.3:template[.3],.6:template[.6]},
        "null":"template_tfidf"
    },
    "Broken Timing":{
        "clean":timing["clean"],
        "conditions":{.1:timing[.1],.3:timing[.3],.6:timing[.6]},
        "null":"timing_structural"
    },
    "Entity Inconsistency":{
        "clean":entity["clean"],
        "conditions":{.1:entity[.1],.3:entity[.3],.6:entity[.6]},
        "null":"campaign_minilm"
    },
    "Missing Cross-Channel Reference":{
        "clean":missing["clean"],
        "conditions":{.1:missing[.1],.3:missing[.3],.6:missing[.6]},
        "null":"campaign_minilm"
    },
    "Over-Polished Language":{
        "clean":op["clean"],
        "conditions":{.1:op[.1],.3:op[.3],.6:op[.6]},
        "null":"op_minilm"
    }
}

In [32]:
for name,cfg in SCENARIOS.items():
    print(name,cfg["clean"].shape,{s:X.shape for s,X in cfg["conditions"].items()})

Template Repetition (300, 5000) {0.1: (300, 5000), 0.3: (300, 5000), 0.6: (300, 5000)}
Broken Timing (300, 13) {0.1: (300, 13), 0.3: (300, 13), 0.6: (300, 13)}
Entity Inconsistency (300, 384) {0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
Missing Cross-Channel Reference (300, 384) {0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
Over-Polished Language (300, 384) {0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}


In [37]:
# run the rbf anchor cells
power_runs=pd.concat([
    run_rbf_power(cfg["clean"],cfg["conditions"],name)
    for name,cfg in SCENARIOS.items()
],ignore_index=True)

power_summary=summarize_rejections(
    power_runs,
    ["scenario","severity"]
)

power_summary

,scenario,severity,rejections,repetitions,power,ci_low,ci_high
0,Broken Timing,0.1,0,200,0.000,0.000000,0.018845
1,Broken Timing,0.3,0,200,0.000,0.000000,0.018845
2,Broken Timing,0.6,19,200,0.095,0.061663,0.143602
3,Entity Inconsistency,0.1,0,200,0.000,0.000000,0.018845
4,Entity Inconsistency,0.3,0,200,0.000,0.000000,0.018845
5,Entity Inconsistency,0.6,1,200,0.005,0.000883,0.027774
6,Missing Cross-Channel Reference,0.1,0,200,0.000,0.000000,0.018845
7,Missing Cross-Channel Reference,0.3,0,200,0.000,0.000000,0.018845
8,Missing Cross-Channel Reference,0.6,32,200,0.160,0.115674,0.217141
9,Over-Polished Language,0.1,0,200,0.000,0.000000,0.018845


In [38]:
# Run the four RBF clean-null calibrations
NULLS={
    "template_tfidf":SCENARIOS["Template Repetition"]["clean"],
    "timing_structural":SCENARIOS["Broken Timing"]["clean"],
    "campaign_minilm":SCENARIOS["Entity Inconsistency"]["clean"],
    "op_minilm":SCENARIOS["Over-Polished Language"]["clean"]
}

null_runs=pd.concat([
    run_rbf_null(clean,name)
    for name,clean in NULLS.items()
],ignore_index=True)

null_summary=summarize_rejections(null_runs,["null"])
null_summary

,null,rejections,repetitions,power,ci_low,ci_high
0,campaign_minilm,8,200,0.040,0.020406,0.076932
1,op_minilm,5,200,0.025,0.010725,0.057178
2,template_tfidf,6,200,0.030,0.013820,0.063894
3,timing_structural,10,200,0.050,0.027383,0.089578


In [39]:
power_detail=power_runs.groupby(["scenario","severity"]).agg(
    power=("reject_H0","mean"),
    mean_accuracy=("accuracy","mean"),
    mean_p=("p_value","mean")
).reset_index()

power_detail.merge(
    power_summary[["scenario","severity","ci_low","ci_high"]],
    on=["scenario","severity"]
)

,scenario,severity,power,mean_accuracy,mean_p,ci_low,ci_high
0,Broken Timing,0.1,0.000,0.51105,4.531110e-01,0.000000,0.018845
1,Broken Timing,0.3,0.000,0.52995,3.175466e-01,0.000000,0.018845
2,Broken Timing,0.6,0.095,0.55435,1.827333e-01,0.061663,0.143602
3,Entity Inconsistency,0.1,0.000,0.50405,5.076927e-01,0.000000,0.018845
4,Entity Inconsistency,0.3,0.000,0.51120,4.523862e-01,0.000000,0.018845
5,Entity Inconsistency,0.6,0.005,0.52685,3.395910e-01,0.000883,0.027774
6,Missing Cross-Channel Reference,0.1,0.000,0.50670,4.867386e-01,0.000000,0.018845
7,Missing Cross-Channel Reference,0.3,0.000,0.52215,3.737271e-01,0.000000,0.018845
8,Missing Cross-Channel Reference,0.6,0.160,0.55445,1.894252e-01,0.115674,0.217141
9,Over-Polished Language,0.1,0.000,0.50670,4.869239e-01,0.000000,0.018845


In [40]:
from pathlib import Path

OUT=Path("week8")
OUT.mkdir(exist_ok=True)

power_runs.to_csv(OUT/"c2st_rbf_n100_power_runs.csv",index=False)
power_summary.to_csv(OUT/"c2st_rbf_n100_power_summary.csv",index=False)
null_runs.to_csv(OUT/"c2st_rbf_n100_null_runs.csv",index=False)
null_summary.to_csv(OUT/"c2st_rbf_n100_null_summary.csv",index=False)

# Run time benchmark

In [20]:
# Check current metric signatures
import inspect
import src.metrics as metrics

for name in [
    "c2st_paired",
    "c2st_rbf_paired",
    "mmd_paired_permutation_test",
    "energy_paired_permutation_test",
    "sinkhorn_paired_permutation_test"
]:
    fn=getattr(metrics,name)
    print(name,inspect.signature(fn))

c2st_paired (P, Q, test_size=0.5, n_permutations=100, seed=42, alpha=0.05)
c2st_rbf_paired (P, Q, test_size=0.5, n_permutations=100, seed=42, alpha=0.05, C=1.0, gamma='scale')
mmd_paired_permutation_test (P, Q, sigma=1.0, n_permutations=100, seed=42, alpha=0.05)
energy_paired_permutation_test (P, Q, n_permutations=100, seed=42, alpha=0.05)
sinkhorn_paired_permutation_test (P, Q, blur, n_permutations=100, seed=42, alpha=0.05, p=2, backend='tensorized')


In [21]:
# runtime benchmark config
import gc
import time
import numpy as np
import pandas as pd
from scipy import sparse
from src.metrics import (
    c2st_paired,
    c2st_rbf_paired,
    mmd_paired_permutation_test,
    energy_paired_permutation_test,
    sinkhorn_paired_permutation_test
)

BENCH_N=[50,100,200]
BENCH_REPS=5
BENCH_SEVERITY=.3
N_PERM=100
ALPHA=.05
BENCH_SEED=91000

SCALES={
    "Template Repetition":{"sigma":1.414213562373095,"blur":.1414213562373095},
    "Broken Timing":{"sigma":4.82953793971741,"blur":.482953793971741},
    "Entity Inconsistency":{"sigma":.4312044032354748,"blur":.043719952430236554},
    "Missing Cross-Channel Reference":{"sigma":.4312044032354748,"blur":.043719952430236554},
    "Over-Polished Language":{"sigma":1.3281240463256836,"blur":.13281227117017558}
}

In [22]:
# flexible metric caller
# Pass only arguments supported by each metric function
def call_supported(fn,P,Q,**kwargs):
    params=inspect.signature(fn).parameters
    return fn(P,Q,**{k:v for k,v in kwargs.items() if k in params})

def dense(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X)

In [23]:
# define 5 benchmark configs
# Run one metric configuration
def run_benchmark_metric(test_config,P,Q,scenario,seed):
    scale=SCALES[scenario]

    if test_config=="C2ST-Linear":
        return call_supported(
            c2st_paired,dense(P),dense(Q),
            seed=seed,random_state=seed,alpha=ALPHA
        )

    if test_config=="C2ST-RBF":
        return c2st_rbf_paired(
            P,Q,seed=seed,alpha=ALPHA
        )

    if test_config=="MMD":
        return call_supported(
            mmd_paired_permutation_test,dense(P),dense(Q),
            sigma=scale["sigma"],
            n_perm=N_PERM,n_permutations=N_PERM,permutations=N_PERM,
            seed=seed,random_state=seed,alpha=ALPHA
        )

    if test_config=="Energy":
        return call_supported(
            energy_paired_permutation_test,dense(P),dense(Q),
            n_perm=N_PERM,n_permutations=N_PERM,permutations=N_PERM,
            seed=seed,random_state=seed,alpha=ALPHA
        )

    if test_config=="Sinkhorn":
        return call_supported(
            sinkhorn_paired_permutation_test,dense(P),dense(Q),
            blur=scale["blur"],
            n_perm=N_PERM,n_permutations=N_PERM,permutations=N_PERM,
            seed=seed,random_state=seed,alpha=ALPHA
        )

    raise ValueError(test_config)

In [24]:
# one run check
scenario="Entity Inconsistency"
cfg=SCENARIOS[scenario]
rng=np.random.default_rng(BENCH_SEED)
idx=rng.choice(cfg["clean"].shape[0],size=50,replace=False)

P=cfg["clean"][idx]
Q=cfg["conditions"][BENCH_SEVERITY][idx]

for test_config in ["C2ST-Linear","C2ST-RBF","MMD","Energy","Sinkhorn"]:
    print(test_config)
    run_benchmark_metric(test_config,P,Q,scenario,BENCH_SEED)

C2ST-Linear
C2ST-RBF
MMD
Energy
Sinkhorn


In [46]:
# Benchmark 5 repetitions per Scenario x n x Test Configuration
TEST_CONFIGS=["C2ST-Linear","C2ST-RBF","MMD","Energy","Sinkhorn"]

rows=[]

for scenario,cfg in SCENARIOS.items():
    Qall=cfg["conditions"][BENCH_SEVERITY]

    for n in BENCH_N:
        for rep in range(BENCH_REPS):
            seed=BENCH_SEED+rep
            idx=np.random.default_rng(seed).choice(cfg["clean"].shape[0],size=n,replace=False)
            P=cfg["clean"][idx]
            Q=Qall[idx]

            for test_config in TEST_CONFIGS:
                gc.collect()
                t0=time.perf_counter()
                run_benchmark_metric(test_config,P,Q,scenario,seed)
                elapsed=time.perf_counter()-t0

                rows.append({
                    "scenario":scenario,
                    "n":n,
                    "test_config":test_config,
                    "repetition":rep,
                    "seconds":elapsed
                })

runtime_runs=pd.DataFrame(rows)

print("Rows:",len(runtime_runs))
runtime_runs.head()

/Users/jessica/fidelity-metric-lab/.venv312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/jessica/fidelity-metric-lab/.venv312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

Rows: 375


,scenario,n,test_config,repetition,seconds
0,Template Repetition,50,C2ST-Linear,0,0.110474
1,Template Repetition,50,C2ST-RBF,0,0.031535
2,Template Repetition,50,MMD,0,0.239226
3,Template Repetition,50,Energy,0,0.053157
4,Template Repetition,50,Sinkhorn,0,4.035860


# prueba

In [32]:
import importlib
import src.metrics as metrics

importlib.reload(metrics)

from src.metrics import c2st_paired

In [33]:
# Locate the exact C2ST-Linear convergence failures
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

convergence_rows=[]

for scenario,cfg in SCENARIOS.items():
    Qall=cfg["conditions"][BENCH_SEVERITY]

    for n in BENCH_N:
        for rep in range(BENCH_REPS):
            seed=BENCH_SEED+rep

            idx=np.random.default_rng(seed).choice(
                cfg["clean"].shape[0],
                size=n,
                replace=False
            )

            P=cfg["clean"][idx]
            Q=Qall[idx]

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always",ConvergenceWarning)

                run_benchmark_metric(
                    "C2ST-Linear",
                    P,
                    Q,
                    scenario,
                    seed
                )

            conv_warnings=[
                w for w in caught
                if issubclass(w.category,ConvergenceWarning)
            ]

            convergence_rows.append({
                "scenario":scenario,
                "severity":BENCH_SEVERITY,
                "n":n,
                "repetition":rep,
                "seed":seed,
                "warning":len(conv_warnings)>0,
                "n_warnings":len(conv_warnings)
            })

convergence_check=pd.DataFrame(convergence_rows)

failures=convergence_check[
    convergence_check["warning"]
].copy()

print("Total runs checked:",len(convergence_check))
print("Runs with convergence warning:",len(failures))

display(failures)

Total runs checked: 75
Runs with convergence warning: 0


,scenario,severity,n,repetition,seed,warning,n_warnings


In [26]:
summary=convergence_check.groupby(
    ["scenario","severity","n"]
)["warning"].agg(
    warnings="sum",
    runs="count"
).reset_index()

summary["warning_rate"]=summary["warnings"]/summary["runs"]

display(
    summary[summary["warnings"]>0]
)

,scenario,severity,n,warnings,runs,warning_rate
0,Broken Timing,0.3,50,2,5,0.4
1,Broken Timing,0.3,100,4,5,0.8
2,Broken Timing,0.3,200,4,5,0.8


In [27]:
# Inspect feature scales for Broken Timing
cfg=SCENARIOS["Broken Timing"]

P=np.asarray(cfg["clean"],dtype=float)
Q=np.asarray(cfg["conditions"][.3],dtype=float)

scale_check=pd.DataFrame({
    "feature":[f"f{i}" for i in range(P.shape[1])],
    "clean_mean":P.mean(axis=0),
    "clean_std":P.std(axis=0),
    "clean_min":P.min(axis=0),
    "clean_max":P.max(axis=0),
    "defect_mean":Q.mean(axis=0),
    "defect_std":Q.std(axis=0),
    "defect_min":Q.min(axis=0),
    "defect_max":Q.max(axis=0)
})

display(scale_check)

,feature,clean_mean,clean_std,clean_min,clean_max,defect_mean,defect_std,defect_min,defect_max
0,f0,6.966667,1.430229,5.000000,9.000000,6.966667,1.430229,5.000000,9.000000
1,f1,365.996667,123.175660,72.000000,708.000000,346.900000,125.233475,72.000000,696.000000
2,f2,61.053056,13.525299,18.000000,92.200000,57.662635,14.390500,18.000000,92.200000
3,f3,61.350000,19.198720,16.000000,110.000000,58.378333,20.324723,10.000000,110.000000
4,f4,29.387991,7.564022,8.154753,47.305814,31.565530,8.022214,8.154753,50.424035
5,f5,2.800000,0.400000,2.000000,3.000000,2.800000,0.400000,2.000000,3.000000
6,f6,0.659258,0.191714,0.125000,1.000000,0.659258,0.191714,0.125000,1.000000
7,f7,0.917491,0.071339,0.591673,1.000000,0.917491,0.071339,0.591673,1.000000
8,f8,0.515776,0.104859,0.333333,0.857143,0.515776,0.104859,0.333333,0.857143
9,f9,59.413028,3.197221,53.000000,67.000000,59.413028,3.197221,53.000000,67.000000


In [28]:
# Check how different the feature scales are
nonzero_std=scale_check.loc[scale_check["clean_std"]>0,"clean_std"]

print("Smallest non-zero std:",nonzero_std.min())
print("Largest std:",nonzero_std.max())
print("Largest / smallest std:",nonzero_std.max()/nonzero_std.min())

Smallest non-zero std: 0.07133928435606887
Largest std: 123.17566042941367
Largest / smallest std: 1726.6175507819632


In [29]:
# Test whether increasing max_iter solves Broken Timing convergence
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from scipy import sparse

rows=[]

cfg=SCENARIOS["Broken Timing"]
Qall=cfg["conditions"][BENCH_SEVERITY]

for _,row in failures.iterrows():
    n=int(row["n"])
    seed=int(row["seed"])

    idx=np.random.default_rng(seed).choice(
        cfg["clean"].shape[0],
        size=n,
        replace=False
    )

    P=cfg["clean"][idx]
    Q=Qall[idx]

    pair_indices=np.arange(n)

    train_idx,test_idx=train_test_split(
        pair_indices,
        test_size=0.5,
        random_state=seed,
        shuffle=True
    )

    P_train=P[train_idx]
    Q_train=Q[train_idx]
    P_test=P[test_idx]
    Q_test=Q[test_idx]

    if sparse.issparse(P):
        X_train=sparse.vstack([P_train,Q_train],format="csr")
        X_test=sparse.vstack([P_test,Q_test],format="csr")
    else:
        X_train=np.vstack([P_train,Q_train])
        X_test=np.vstack([P_test,Q_test])

    y_train=np.r_[
        np.zeros(len(train_idx),dtype=int),
        np.ones(len(train_idx),dtype=int)
    ]

    y_test=np.r_[
        np.zeros(len(test_idx),dtype=int),
        np.ones(len(test_idx),dtype=int)
    ]

    for max_iter in [1000,5000,10000]:

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always",ConvergenceWarning)

            clf=LogisticRegression(
                random_state=seed,
                max_iter=max_iter
            )

            clf.fit(X_train,y_train)
            pred=clf.predict(X_test)

        conv_warning=any(
            issubclass(w.category,ConvergenceWarning)
            for w in caught
        )

        rows.append({
            "n":n,
            "seed":seed,
            "max_iter":max_iter,
            "warning":conv_warning,
            "n_iter":int(clf.n_iter_[0]),
            "accuracy":float((pred==y_test).mean())
        })

iter_diagnostic=pd.DataFrame(rows)

display(iter_diagnostic)

,n,seed,max_iter,warning,n_iter,accuracy
0,50,91001,1000,True,1000,0.540
1,50,91001,5000,False,1261,0.540
2,50,91001,10000,False,1261,0.540
3,50,91002,1000,True,1000,0.540
4,50,91002,5000,False,1572,0.540
5,50,91002,10000,False,1572,0.540
6,100,91000,1000,True,1000,0.580
7,100,91000,5000,False,1944,0.580
8,100,91000,10000,False,1944,0.580
9,100,91001,1000,True,1000,0.550


In [30]:
display(
    iter_diagnostic.groupby("max_iter").agg(
        warnings=("warning","sum"),
        mean_n_iter=("n_iter","mean"),
        max_n_iter=("n_iter","max"),
        mean_accuracy=("accuracy","mean")
    ).reset_index()
)

,max_iter,warnings,mean_n_iter,max_n_iter,mean_accuracy
0,1000,10,1000.0,1000,0.5715
1,5000,0,1875.6,2633,0.5720
2,10000,0,1875.6,2633,0.5720


These results are clear: `max_iter=5000` solves the problem without needing to change the scaling.
With 1,000 iterations, there were 10 warnings; with 5,000, there were zero; and with 10,000, you obtained exactly the same `n_iter` as with 5,000, indicating that the solver had already converged before reaching the 5,000 mark. Furthermore, the accuracy barely changes between the two runs, and the average shifts from 0.5715 to 0.5720. This confirms that the issue was primarily that `lbfgs` required more iterations, not that the representation itself was unusable.

# fin de la prueba

In [47]:
# Runtime summary
runtime_summary=runtime_runs.groupby(
    ["scenario","n","test_config"]
)["seconds"].agg(
    mean_seconds="mean",
    median_seconds="median",
    std_seconds="std",
    min_seconds="min",
    max_seconds="max"
).reset_index()

runtime_summary

,scenario,n,test_config,mean_seconds,median_seconds,std_seconds,min_seconds,max_seconds
0,Broken Timing,50,C2ST-Linear,0.229326,0.235444,0.116894,0.105734,0.366161
1,Broken Timing,50,C2ST-RBF,0.004759,0.004348,0.001237,0.003145,0.006231
2,Broken Timing,50,Energy,0.036749,0.023020,0.036227,0.014573,0.101243
3,Broken Timing,50,MMD,0.030060,0.017311,0.030281,0.014132,0.084171
4,Broken Timing,50,Sinkhorn,1.353450,1.296116,0.212358,1.193047,1.719795
...,...,...,...,...,...,...,...,...
70,Template Repetition,200,C2ST-Linear,0.057667,0.054349,0.011907,0.045171,0.076973
71,Template Repetition,200,C2ST-RBF,0.021367,0.020905,0.002220,0.018965,0.025012
72,Template Repetition,200,Energy,0.169034,0.168729,0.006004,0.162165,0.177342
73,Template Repetition,200,MMD,0.179685,0.181409,0.008025,0.166315,0.186320


In [48]:
# Compare median runtime across sample sizes
runtime_pivot=runtime_summary.pivot_table(
    index=["scenario","test_config"],
    columns="n",
    values="median_seconds"
).reset_index()

runtime_pivot

n,scenario,test_config,50,100,200
0,Broken Timing,C2ST-Linear,0.235444,0.362574,0.310590
1,Broken Timing,C2ST-RBF,0.004348,0.005830,0.009885
2,Broken Timing,Energy,0.023020,0.047166,0.116569
3,Broken Timing,MMD,0.017311,0.056456,0.140163
4,Broken Timing,Sinkhorn,1.296116,1.587076,2.254813
5,Entity Inconsistency,C2ST-Linear,0.011419,0.012335,0.012664
6,Entity Inconsistency,C2ST-RBF,0.007465,0.011241,0.029006
7,Entity Inconsistency,Energy,0.020680,0.040534,0.118239
8,Entity Inconsistency,MMD,0.022821,0.045862,0.128609
9,Entity Inconsistency,Sinkhorn,0.930046,1.132995,1.783705


In [49]:
# Save benchmark artifacts
OUT=W7/"week8" if not (W7/"week8").exists() else W7/"week8"
OUT.mkdir(parents=True,exist_ok=True)

runtime_runs.to_csv(OUT/"runtime_benchmark_runs.csv",index=False)
runtime_summary.to_csv(OUT/"runtime_benchmark_summary.csv",index=False)
runtime_pivot.to_csv(OUT/"runtime_benchmark_pivot.csv",index=False)

print("Saved to:",OUT)

Saved to: /Users/jessica/fidelity-metric-lab/results/week7/week8


In [18]:
import importlib
import src.metrics as metrics

importlib.reload(metrics)

from src.metrics import c2st_rbf_paired

cfg=SCENARIOS["Entity Inconsistency"]
P=cfg["clean"][:100]
Q=cfg["conditions"][.3][:100]

c2st_rbf_paired(P,Q,n_permutations=100,seed=42)

{'accuracy': 0.5,
 'p_value': 0.7920792079207921,
 'reject_H0': False,
 'n_pairs': 100,
 'n_train_pairs': 50,
 'n_test_pairs': 50,
 'n_test': 100,
 'n_permutations': 100,
 'seed': 42,
 'C': 1.0,
 'gamma': 'scale'}